# Notebook 4 — Optimisation & Déploiement
## ImmoPredict SN — GridSearch + Validation finale
---

In [ ]:
import pandas as pd
import numpy as np
import json, joblib, os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from scipy.stats import randint, uniform

GOLD='#C9A84C'; NAVY='#0F2444'; GREEN='#0E6B4A'
np.random.seed(42)

# ── Charger la config produite par NB2 ──────────────────────────
with open('../properties/ml/features_config.json') as f:
    cfg = json.load(f)

# NB2 écrit NUM_FEATURES et CAT_FEATURES
df = pd.read_csv('../data/dataset_features.csv')
df = df.dropna(subset=['log_price'])   # log_price créé dans NB2

avail_num = [f for f in cfg['NUM_FEATURES'] if f in df.columns]
avail_cat = [f for f in cfg['CAT_FEATURES'] if f in df.columns]
ALL_FEATS = avail_num + avail_cat

X = df[ALL_FEATS]
y = df['log_price'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'✅ {len(df):,} annonces | Train:{len(X_train):,} Test:{len(X_test):,}')

In [ ]:
# ── Préprocesseur commun ──────────────────────────────────────
num_pipe = Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler())])
cat_pipe = Pipeline([('i',SimpleImputer(strategy='constant',fill_value='Autre')),
                     ('e',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1))])
prep = ColumnTransformer([('n',num_pipe,avail_num),('c',cat_pipe,avail_cat)])

def evaluate(name, y_true, y_pred_log):
    yt = np.expm1(y_true)
    yp = np.expm1(y_pred_log)
    r2   = r2_score(y_true, y_pred_log)
    mae  = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mape = np.mean(np.abs((yt - yp) / yt.clip(1))) * 100
    p20  = np.mean(np.abs((yt - yp) / yt.clip(1)) <= 0.20) * 100
    p30  = np.mean(np.abs((yt - yp) / yt.clip(1)) <= 0.30) * 100
    print(f'{name}: R²={r2:.4f} MAPE={mape:.1f}% ±20%={p20:.1f}% MAE={mae/1e6:.1f}M FCFA')
    return dict(r2=r2,mae=mae,rmse=rmse,mape=mape,pct20=p20,pct30=p30)

print('✅ Préprocesseur prêt')

In [ ]:
# ══════════════════════════════════════════════════════════════
# OPTIMISATION RANDOM FOREST — RandomizedSearchCV (200 itérations)
# Meilleur modèle identifié dans NB3
# ══════════════════════════════════════════════════════════════
from sklearn.ensemble import RandomForestRegressor

param_dist_rf = {
    'model__n_estimators':      randint(200, 1000),
    'model__max_depth':         [None, 10, 15, 20, 25, 30, 40],
    'model__min_samples_leaf':  randint(1, 15),
    'model__min_samples_split': randint(2, 15),
    'model__max_features':      ['sqrt', 'log2', 0.3, 0.5, 0.7],
    'model__max_samples':       uniform(0.6, 0.4),       # bagging
    'model__bootstrap':         [True],
    'model__min_impurity_decrease': uniform(0, 0.01),
}

rf_pipe = Pipeline([
    ('prep', prep),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

search_rf = RandomizedSearchCV(
    rf_pipe, param_dist_rf,
    n_iter=200, cv=5, scoring='r2',
    random_state=42, n_jobs=-1, verbose=1,
)
print('Optimisation Random Forest (200 itérations × 5 folds)...')
search_rf.fit(X_train, y_train)
print(f'\n✅ Meilleur R² CV : {search_rf.best_score_:.4f}')
print(f'Meilleurs params  :')
for k, v in search_rf.best_params_.items():
    print(f'  {k}: {v}')
res_rf = evaluate('RandomForest Opt.', y_test, search_rf.predict(X_test))


In [ ]:
# ══════════════════════════════════════════════════════════════
# ANALYSE DES RÉSULTATS — CV scores distribution
# ══════════════════════════════════════════════════════════════
import pandas as pd

# Distribution des scores CV
cv_results = pd.DataFrame(search_rf.cv_results_)
top20 = cv_results.nlargest(20, 'mean_test_score')[
    ['mean_test_score', 'std_test_score',
     'param_model__n_estimators', 'param_model__max_depth',
     'param_model__min_samples_leaf', 'param_model__max_features']
].reset_index(drop=True)

print('Top 20 configurations:')
print(top20.to_string())

# Visualiser la distribution des scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(cv_results['mean_test_score'], bins=40, color='#C9A84C', edgecolor='white', lw=0.3)
axes[0].axvline(search_rf.best_score_, color='#0F2444', lw=2, ls='--',
                label=f'Meilleur: {search_rf.best_score_:.4f}')
axes[0].set_title('Distribution des R² CV (200 configs)', fontweight='bold')
axes[0].set_xlabel('R² moyen CV'); axes[0].legend()

# Impact de max_depth sur R²
depth_scores = cv_results.groupby('param_model__max_depth')['mean_test_score'].mean()
axes[1].bar([str(d) for d in depth_scores.index], depth_scores.values, color='#0E6B4A')
axes[1].set_title('R² moyen par max_depth', fontweight='bold')
axes[1].set_xlabel('max_depth'); axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('rf_cv_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Meilleur modèle = Random Forest optimisé ─────────────────
best_name = 'RandomForest Opt.'
best_opt  = search_rf.best_estimator_
best_res  = res_rf

print(f'🏆 Modèle final : {best_name}')
print(f'   R²   = {best_res["r2"]:.4f}')
print(f'   MAPE = {best_res["mape"]:.1f}%')
print(f'   ±20% = {best_res["pct20"]:.1f}%')
print(f'   ±30% = {best_res["pct30"]:.1f}%')

# Feature importance RF
rf_model    = best_opt.named_steps['model']
feat_names  = avail_num + avail_cat
importances = rf_model.feature_importances_

imp_df = (pd.DataFrame({'feature': feat_names[:len(importances)],
                        'importance': importances})
           .sort_values('importance', ascending=False)
           .head(20))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1], color='#C9A84C')
ax.set_title('Top 20 Features — Random Forest Optimisé', fontweight='bold')
ax.set_xlabel('Importance (MDI)')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 features:')
for _, row in imp_df.head(10).iterrows():
    bar = '█' * int(row['importance'] / importances.max() * 30)
    print(f'  {row["feature"]:25} {bar} {row["importance"]:.4f}')


In [ ]:
# ── Learning curves ───────────────────────────────────────────
train_sizes, train_scores, val_scores = learning_curve(
    best_opt, X, y, cv=5,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='r2', n_jobs=-1
)

plt.figure(figsize=(10, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), color=GOLD, label='Train', linewidth=2)
plt.fill_between(train_sizes,
                 train_scores.mean(axis=1)-train_scores.std(axis=1),
                 train_scores.mean(axis=1)+train_scores.std(axis=1),
                 alpha=0.15, color=GOLD)
plt.plot(train_sizes, val_scores.mean(axis=1), color=NAVY, label='Validation', linewidth=2)
plt.fill_between(train_sizes,
                 val_scores.mean(axis=1)-val_scores.std(axis=1),
                 val_scores.mean(axis=1)+val_scores.std(axis=1),
                 alpha=0.15, color=NAVY)
plt.axhline(y=0.85, color=GREEN, linestyle='--', linewidth=1.5, label='Cible R²=0.85')
plt.xlabel('Taille du jeu d\'entraînement')
plt.ylabel('R²')
plt.title(f'Learning Curves — {best_name}', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Test sur 7 cas réels ──────────────────────────────────────
# Ces cas représentent des types de biens typiques du marché sénégalais
print('=== TESTS SUR CAS RÉELS ===')
cas_tests = [
    {'description': 'Villa 4ch Almadies',       'city':'Almadies',  'bedrooms':4, 'surface_area':300, 'expected': '200-400M FCFA'},
    {'description': 'Appt 3ch Ouakam',           'city':'Ouakam',    'bedrooms':3, 'surface_area':120, 'expected': '40-80M FCFA'},
    {'description': 'Studio Plateau',            'city':'Plateau',   'bedrooms':1, 'surface_area':45,  'expected': '15-35M FCFA'},
    {'description': 'Terrain 500m² Pikine',      'city':'Pikine',    'bedrooms':0, 'surface_area':500, 'expected': '5-20M FCFA'},
    {'description': 'Duplex 5ch VDN',            'city':'VDN',       'bedrooms':5, 'surface_area':250, 'expected': '80-160M FCFA'},
    {'description': 'Maison 2ch Rufisque',        'city':'Rufisque',  'bedrooms':2, 'surface_area':80,  'expected': '5-15M FCFA'},
    {'description': 'Appt 4ch Mermoz Standing', 'city':'Mermoz',    'bedrooms':4, 'surface_area':200, 'expected': '100-200M FCFA'},
]

# Charger predict.py pour les tests
import sys
sys.path.insert(0, '../properties/ml')
try:
    from predict import predict_price
    for cas in cas_tests:
        res = predict_price(
            city=cas['city'],
            property_type='Villa' if cas['bedrooms']>3 else 'Appartement',
            surface_area=cas['surface_area'],
            bedrooms=cas['bedrooms'],
        )
        pred_M = res['predicted_price'] / 1e6
        print(f'  {cas["description"]:35} → {pred_M:.0f}M FCFA  (attendu: {cas["expected"]})')
except ImportError as e:
    print(f'predict.py non disponible: {e}')
    print('Assurez-vous que NB3 a bien généré model.pkl')

In [ ]:
# ── Sauvegarder le modèle optimisé final ─────────────────────
os.makedirs('../properties/ml', exist_ok=True)

joblib.dump({
    'pipeline':             best_opt,
    'best_model_name':      best_name,
    'search_method':        'RandomizedSearchCV 200 iter',
    'best_params':          {str(k): str(v) for k, v in search_rf.best_params_.items()},
    'features':             avail_num + avail_cat,
    'numeric_features':     avail_num,
    'categorical_features': avail_cat,
    'metrics': {
        'r2':    round(best_res['r2'],   4),
        'mae':   round(best_res['mae'],  2),
        'rmse':  round(best_res['rmse'], 2),
        'mape':  round(best_res['mape'], 2),
        'pct20': round(best_res['pct20'],1),
        'pct30': round(best_res['pct30'],1),
    },
}, '../properties/ml/model.pkl')

with open('../properties/ml/results.json', 'w') as f:
    json.dump({
        'best_model': best_name,
        'metrics': {k: round(float(v), 4) if isinstance(v, float) else v
                    for k, v in best_res.items()},
    }, f, indent=2)

print('\n' + '='*60)
print('  RÉCAPITULATIF FINAL — ImmoPredict SN')
print('='*60)
print(f'  Modèle    : {best_name}')
print(f'  R²        : {best_res["r2"]:.4f}')
print(f'  MAE       : {best_res["mae"]/1e6:.1f}M FCFA')
print(f'  MAPE      : {best_res["mape"]:.1f}%')
print(f'  ±20%      : {best_res["pct20"]:.1f}%')
print(f'  ±30%      : {best_res["pct30"]:.1f}%')
print()
print('  Fichiers sauvegardés :')
print('    properties/ml/model.pkl')
print('    properties/ml/results.json')
print('='*60)
